![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/streamlit_notebooks/BertForSequenceClassification.ipynb)

# `BertForSequenceClassification` **Models**

## 1. Colab Setup

In [1]:
!pip -q uninstall -y pyspark spark-nlp sparknlp dataproc-spark-connect
!pip -q install pyspark==3.4.4 spark-nlp==6.3.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.2/746.2 kB 53.3 MB/s eta 0:00:00


Import Libraries

In [2]:
import json
import pandas as pd
import numpy as np

import sparknlp
import pyspark.sql.functions as F

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from sparknlp.annotator import *
from sparknlp.base import *
from sparknlp.pretrained import PretrainedPipeline
from pyspark.sql.types import StringType, IntegerType

## 2. Start Spark Session

In [3]:
spark = sparknlp.start()

print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

spark

Spark NLP version 6.3.2
Apache Spark version: 3.4.4


## 3. Select the DL model

In [4]:
### Select Model
model_antisemitism = 'bert_sequence_classifier_antisemitism'
model_trec_coarse = "bert_sequence_classifier_trec_coarse"
model_age_news = "bert_sequence_classifier_age_news"
model_hatexplain = "bert_sequence_classifier_hatexplain"
model_emotion = "bert_sequence_classifier_emotion"
model_banking = "bert_sequence_classifier_banking77"

## 4. Some sample examples

In [ ]:
text_antisemitism = ["""Shylock in Merchant of Venice. Shylock was a Jew and moneylender. Depends on the context it is used but as the antisemitism is hotly debated nowadays if I were a Jew I wouldn't like to hear it. Perhaps I'm wrong but that's my opinion.""",
"""That Jew gripped yo nuts and you did nothing and you been attacking black people ever since. Probably like that shit""",
"""They came for the Jews, and I did not speak out Because I was not a Jew.Then they came for me and there was no one left to speak for me""",
"""David is a sephardic jew huh.... now i have to give him my entire heart i guess""",
"""I asked a genuine question, she has been smearing @georgegalloway for a while now without any evidence. Am I not allowed to ask her to show me the anti Semitism. Remember @RachelRileyRR is the one who said she ‘doesn’t look like a normal jew’ that to me is anti Semitic""",
"""I pointed the finger directly at the fascists still in control of Europe the muh jew shills began in earnest. Distraction, anger, insult, lies.""",
]

In [ ]:
text_trec_coarse = ["""Germany is the largest country in Europe economically.""",
"""What other prince showed his paintings in a two-prince exhibition with Prince Charles in London?""",
"""What is the name of the chronic neurological autoimmune disease which attacks the protein sheath that surrounds nerve cells causing a gradual loss of movement in the body?""",
"""How many hands does Bjorn Borg use when hitting his forehand?""",
"""CNN is the abbreviation for what?""",
"""Give a reason for American Indians oftentimes dropping out of school.""",
"""What was organized as a confederate veterans social club in Pulaski, in Tennessee, in 1866?""",
"""Who was the first person inducted into the U.S. Swimming Hall of Fame?""",
"""What did the only repealed amendment to the U.S. Constitution deal with?""",
]

In [ ]:
text_age_news = ["""Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group, which has a reputation for making well-timed and occasionally controversial plays in the defense industry, has quietly placed its bets on another part of the market.""",
"""Juventus and Olympique Lyon joined Chelsea and Inter Milan in the last 16 of the Champions League after both scored late goals to seal their places on Wednesday.""",
"""Fearing the fate of Italy, the centre-right government has threatened to be merciless with those who flout tough restrictions.""",
"""Cisco Systems is teaming up with researchers at the Massachusetts Institute of Technology to develop new communications technology that the company hopes to ultimately turn into products.""",
"""Golf is not a sport renowned for being at ease with gender issues. Many of the games great clubs refuse to admit women, and the occasional inclusion of women in men professional tournaments has caused consternation from Arran to Augusta.""",
"""The Supreme Council of the United Arab Emirates (UAE) on Wednesday evening unanimously elected Sheikh Khalifa bin Zayed Al Nahyan as the new president of the UAE, the official Emirates news agency WAM reported Wednesday.""",
"""The Norwegian hacker famed for developing DVD encryption cracking software has apparently struck again; this time breaking the locks on Apple Computer Inc.'s wireless music streaming technology.""",
"""Utility holding company Ameren Corp. announced plans Wednesday to reduce its payroll by offering 950 union and management employees financial incentives to leave."""
]

In [ ]:
text_hatexplain = ["""If English is not imposition then Hindi is also not imposition shut up chutiya retards stop hindi imposition.""",
"""The uk has threatened to return radioactive waste to the eu if an agreement cannot be reached can not we keep the radioactive waste and send back all the paki migrants instead""",
"""Afghani immigrant once told me that in Afghanistan we do not believe in fences while our conversation was about livestock guardian dogs it now occurs to me what this really means is that there no belief in protection of persons or property""",
"""I post a video showing the shocking promotion of violence by the south african eff themselves and jewtube still say i am promoting violence""",
"""I want to go back to school and get an education but i am poor and colleges are kike rip offs i am not satisfied with where i am in life and honestly have no clue where or how to start pushing forward""",
"""I am confused how a president blocked stimulus checks for those who married to immigrants while he himself married to an immigrant""",
"""I love you very much!"""]

In [6]:
text_emotion = ["The new bike lanes on Main Street are fantastic! I feel so much safer cycling to work now.","I've been waiting 3 months for the pothole on Oak Avenue to be fixed. Someone is going to get hurt!","My kids love the new playground equipment at Lincoln Park. Thank you for listening to the community!","The construction noise at 6 AM every morning is making me lose sleep. This has been going on for weeks.","Why is the city spending millions on a stadium when our schools need repairs? This makes me angry.","I'm worried about the lack of street lighting near the high school. It's not safe for students walking home.","The new 311 app is so confusing. I tried for an hour to submit a request and couldn't figure it out.","Thank you to the snow removal crew! They did an amazing job clearing our street quickly."]

In [ ]:
text_banking = ["""I have been waiting over a week. Is the card still coming?""",
"""I need a transaction reversed from my account.""",
"""How long does it take for cards to be delivered after ordering them?""",
"""I've just been married and need to update my name""",
"""I'm interested in learning more about disposable virtual cards.	""",
"""I tried topping up using my card, but the money is gone?"""]

## 5. Define Spark NLP pipeline

In [7]:
model_dict = {model_emotion: text_emotion}

In [10]:
def run_pipeline(model, text, results):
  document_assembler = DocumentAssembler() \
      .setInputCol('text') \
      .setOutputCol('document')

  tokenizer = Tokenizer() \
      .setInputCols(['document']) \
      .setOutputCol('token')

  sequenceClassifier = BertForSequenceClassification\
        .pretrained(model, 'en') \
        .setInputCols(['token', 'document']) \
        .setOutputCol('pred_class')

  pipeline = Pipeline(
      stages=[
          document_assembler,
          tokenizer,
          sequenceClassifier])

  df = spark.createDataFrame(text, StringType()).toDF("text")
  results[model]=(pipeline.fit(df).transform(df))

## 6. Run the pipeline

In [11]:
results = {}
for model, text in zip(model_dict.keys(),model_dict.values()):
  run_pipeline(model, text, results)

bert_sequence_classifier_emotion download started this may take some time.
Approximate size to download 391.1 MB
[OK!]


## 7. Visualize results

In [12]:
for model_name, result in zip(results.keys(),results.values()):
  res = result.select(F.explode(F.arrays_zip(result.document.result,
                                             result.pred_class.result,
                                             result.pred_class.metadata)).alias("col"))\
              .select(F.expr("col['1']").alias("prediction"),
                      F.expr("col['2']").alias("confidence"),
                      F.expr("col['0']").alias("sentence"))

  if res.count()>1:
    udf_func = F.udf(lambda x,y:  x[str(y)])
    print("\n",model_name,"\n")
    res.withColumn('confidence', udf_func(res.confidence, res.prediction)).show(truncate=False)
    print("\n**********************************\n")


 bert_sequence_classifier_emotion 

+----------+----------+------------------------------------------------------------------------------------------------------------+
|prediction|confidence|sentence                                                                                                    |
+----------+----------+------------------------------------------------------------------------------------------------------------+
|joy       |0.99636316|The new bike lanes on Main Street are fantastic! I feel so much safer cycling to work now.                  |
|sadness   |0.96886003|I've been waiting 3 months for the pothole on Oak Avenue to be fixed. Someone is going to get hurt!         |
|joy       |0.94965917|My kids love the new playground equipment at Lincoln Park. Thank you for listening to the community!        |
|anger     |0.62458676|The construction noise at 6 AM every morning is making me lose sleep. This has been going on for weeks.     |
|anger     |0.9934534 |Why is th

In [13]:
import sys, json, time
print("TEST_ENVIRONMENT", sys.version.split()[0], sparknlp.version(), spark.version)
# Same NER stages and default model as the official NER_EN notebook.
text_list = ["There's a broken streetlight at the corner of 5th Avenue and Elm Street. It's been out since January 15th.", 'The sidewalk on Washington Boulevard between Park Street and City Hall is cracked and dangerous.', 'Water main break on Grove Avenue near Roosevelt Elementary School. Reported at 8:00 AM today.', 'The traffic signal at Main Street and Oak Avenue has been stuck on red since Tuesday morning.', 'Graffiti on the Madison Park Community Center needs removal. It appeared over the weekend.', 'Tree branches over Central Avenue near house number 482 are blocking street signs.', 'The bus shelter at Maple Street and Route 42 has a leaking roof. Passengers are getting wet.', 'Potholes on Lincoln Highway between mile markers 15 and 18 need urgent repair.']
ner_started = time.time()
documentAssembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")
embeddings = WordEmbeddingsModel.pretrained("glove_100d").setInputCols(["document", "token"]).setOutputCol("embeddings")
ner_model = NerDLModel.pretrained("onto_100", "en").setInputCols(["document", "token", "embeddings"]).setOutputCol("ner")
ner_converter = NerConverter().setInputCols(["document", "token", "ner"]).setOutputCol("ner_chunk")
nlp_pipeline = Pipeline(stages=[documentAssembler, tokenizer, embeddings, ner_model, ner_converter])
df = spark.createDataFrame(text_list, StringType()).toDF("text")
ner_result = nlp_pipeline.fit(df).transform(df)
ner_rows = ner_result.select("text", "ner_chunk").collect()
ner_payload = [{"report": i+1, "text": row.text, "entities": [{"text": e.result, "label": e.metadata.get("entity"), "begin": e.begin, "end": e.end} for e in row.ner_chunk]} for i, row in enumerate(ner_rows)]
print("NER_RESULTS_JSON", json.dumps(ner_payload))
print("NER_ELAPSED_SECONDS", round(time.time()-ner_started, 1))
# Display all eight reports rather than only result.collect()[0].
for row in ner_payload:
    print("REPORT", row["report"], row["entities"])


TEST_ENVIRONMENT 3.11.13 6.3.2 3.4.4
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
onto_100 download started this may take some time.
Approximate size to download 13.5 MB
[OK!]
NER_RESULTS_JSON [{"report": 1, "text": "There's a broken streetlight at the corner of 5th Avenue and Elm Street. It's been out since January 15th.", "entities": [{"text": "5th Avenue", "label": "FAC", "begin": 46, "end": 55}, {"text": "Elm Street", "label": "FAC", "begin": 61, "end": 70}, {"text": "January 15th", "label": "DATE", "begin": 93, "end": 104}]}, {"report": 2, "text": "The sidewalk on Washington Boulevard between Park Street and City Hall is cracked and dangerous.", "entities": [{"text": "Washington Boulevard", "label": "FAC", "begin": 16, "end": 35}, {"text": "Park Street and City Hall", "label": "FAC", "begin": 45, "end": 69}]}, {"report": 3, "text": "Water main break on Grove Avenue near Roosevelt Elementary School. Reported at 8:00 AM today.", "e

In [14]:
# Same TREC model and stages as the official CLASSIFICATION_EN_TREC notebook.
classification_started = time.time()
service_text_list = ['The bus stop shelter on Route 42 near Maple Street is damaged. The roof is leaking.', 'Bulk trash pickup request - items have been on curb for a week.', 'Request to add more benches in Lincoln Park. Existing ones are always occupied.', "Parking meter on 3rd Street is malfunctioning and won't accept payment.", 'Noise complaint about River Street bar - loud music past midnight on weekdays.', 'Crosswalk needed at School Road and Highland Avenue intersection. Children crossing unsafely.', 'Public library Wi-Fi has been down for three days. Many rely on this access.', 'Stray dog spotted multiple times in Jefferson neighborhood. Public safety concern.', 'Street signs at the corner of Oak and Elm are completely faded and unreadable.', 'Request for community meeting about proposed zoning changes in downtown district.']
use = UniversalSentenceEncoder.pretrained(lang="en").setInputCols(["document"]).setOutputCol("sentence_embeddings")
document_classifier = ClassifierDLModel.pretrained("classifierdl_use_trec50", "en").setInputCols(["sentence_embeddings"]).setOutputCol("class_")
cls_pipeline = Pipeline(stages=[documentAssembler, use, document_classifier])
service_df = spark.createDataFrame(service_text_list, StringType()).toDF("text")
cls_result = cls_pipeline.fit(service_df).transform(service_df)
cls_payload = [{"request": i+1, "text": row.text, "labels": [x.result for x in row.class_]} for i, row in enumerate(cls_result.select("text", "class_").collect())]
print("CLASS_RESULTS_JSON", json.dumps(cls_payload))
print("CLASS_ELAPSED_SECONDS", round(time.time()-classification_started,1))


tfhub_use download started this may take some time.
Approximate size to download 923.7 MB
[OK!]
classifierdl_use_trec50 download started this may take some time.
Approximate size to download 21.2 MB
[OK!]
CLASS_RESULTS_JSON [{"request": 1, "text": "The bus stop shelter on Route 42 near Maple Street is damaged. The roof is leaking.", "labels": [" LOC_other"]}, {"request": 2, "text": "Bulk trash pickup request - items have been on curb for a week.", "labels": [" NUM_count"]}, {"request": 3, "text": "Request to add more benches in Lincoln Park. Existing ones are always occupied.", "labels": [" LOC_other"]}, {"request": 4, "text": "Parking meter on 3rd Street is malfunctioning and won't accept payment.", "labels": [" ENTY_other"]}, {"request": 5, "text": "Noise complaint about River Street bar - loud music past midnight on weekdays.", "labels": [" DESC_desc"]}, {"request": 6, "text": "Crosswalk needed at School Road and Highland Avenue intersection. Children crossing unsafely.", "labels": 

In [15]:
!pip install --upgrade -q spark-nlp-display
from sparknlp_display import NerVisualizer
print("ORIGINAL_VISUALIZER_FIRST_ROW_ONLY")
NerVisualizer().display(result=ner_result.collect()[0], label_col="ner_chunk", document_col="document")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 12.6 MB/s eta 0:00:00
ORIGINAL_VISUALIZER_FIRST_ROW_ONLY
